# NexaTel Phase 1: Data Loading, Cleaning and Quality Report
I wrote this notebook to run from top to bottom. Every number in the final report table is
**computed by this notebook**, not typed by hand.

## 1. Setup: connect Drive, find the data, import tools

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob, zipfile
import pandas as pd

BASE = "/content/drive/MyDrive/NexaTel/"
DATA = BASE + "data/"

# Unzip only if the CSVs are not already there (so re-running never repeats it)
if not os.path.exists(DATA + "customers.csv"):
    zip_path = glob.glob(BASE + "*.zip")[0]
    zipfile.ZipFile(zip_path).extractall(DATA)

print(len(glob.glob(DATA + "*.csv")), "CSV files found")

Mounted at /content/drive
29 CSV files found


## 2. Load and profile all 24 tables
Project brief, steps 1-3: load every CSV, then report rows, columns and blanks for each table.

In [2]:
tables = {}
for path in sorted(glob.glob(DATA + "*.csv")):
    name = os.path.basename(path).replace(".csv", "")
    if name.endswith("_clean"):          # skip the clean files this notebook saves later
        continue
    tables[name] = pd.read_csv(path)

profile = pd.DataFrame({
    "rows":               {n: t.shape[0] for n, t in tables.items()},
    "columns":            {n: t.shape[1] for n, t in tables.items()},
    "columns_with_nulls": {n: int(t.isnull().any().sum()) for n, t in tables.items()},
})
profile["duplicate_rows"]  = pd.Series({n: int(t.duplicated().sum()) for n, t in tables.items()})
profile["numeric_columns"] = pd.Series({n: int(t.select_dtypes("number").shape[1]) for n, t in tables.items()})
profile["text_columns"]    = profile["columns"] - profile["numeric_columns"]
print(len(tables), "tables loaded")
profile

24 tables loaded


,rows,columns,columns_with_nulls,duplicate_rows,numeric_columns,text_columns
billing,115313,9,0,344,3,6
cities,68,8,0,0,1,7
complaints,16584,7,1,0,0,7
contracts,10607,9,0,0,1,8
customer_feedback,28499,7,0,0,2,5
customers,19076,25,4,76,5,20
data_quality_issue_log,20,3,0,0,1,2
devices,19000,8,0,0,2,6
employees,800,9,0,0,0,9
marketing_campaigns,40,9,0,0,3,6


In [3]:
# Every column that has blanks, in every table, with the % of rows affected
null_rows = []
for n, t in tables.items():
    for col in t.columns:
        k = int(t[col].isnull().sum())
        if k > 0:
            null_rows.append({"table": n, "column": col, "null_count": k,
                              "null_pct": round(k / len(t) * 100, 2)})
null_report = pd.DataFrame(null_rows)
null_report

,table,column,null_count,null_pct
0,complaints,resolution_date,7348,44.31
1,customers,email,745,3.91
2,customers,address,380,1.99
3,customers,annual_income_inr,954,5.00
4,customers,churn_date,15560,81.57
5,subscriptions,plan_id,41,0.20
6,subscriptions,end_date,17017,82.92
7,support_tickets,resolution_hours,13366,42.45
8,support_tickets,resolution_date,13366,42.45


In [4]:
# Date columns are stored as text. List them so we know which ones to convert.
text_date_cols = {}
for n, t in tables.items():
    cols = [c for c in t.columns
            if "date" in c.lower() and not pd.api.types.is_numeric_dtype(t[c])]
    if cols:
        text_date_cols[n] = cols
text_date_cols

{'billing': ['billing_date', 'due_date'],
 'complaints': ['complaint_date', 'resolution_date'],
 'contracts': ['start_date', 'end_date'],
 'customer_feedback': ['feedback_date'],
 'customers': ['date_of_birth', 'acquisition_date', 'churn_date'],
 'devices': ['purchase_date'],
 'employees': ['hire_date'],
 'marketing_campaigns': ['start_date', 'end_date'],
 'payments': ['payment_date'],
 'plan_history': ['change_date'],
 'recharges': ['recharge_date'],
 'retention_campaigns': ['contact_date'],
 'stores': ['opened_date'],
 'subscriptions': ['start_date', 'end_date'],
 'support_tickets': ['created_date', 'resolution_date']}

## 3. Cleaning
`found` collects every issue count we measure. The comparison with the issue log at the end is built from it.

In [5]:
found = {}     # (table, issue_type) -> how many rows we measured

### 3A. customers

In [6]:
customers = tables["customers"].copy()
print("Rows before:", len(customers))

# Duplicates: rows identical in every column
found[("customers", "duplicate_customers")] = int(customers.duplicated().sum())
customers = customers.drop_duplicates()
print("Rows after:", len(customers), "| duplicate customer_ids left:", customers["customer_id"].duplicated().sum())

Rows before: 19076
Rows after: 19000 | duplicate customer_ids left: 0


In [7]:
# Blanks, measured on the 19,000 real customers (after duplicates are gone)
found[("customers", "missing_email")]   = int(customers["email"].isnull().sum())
found[("customers", "missing_address")] = int(customers["address"].isnull().sum())
found[("customers", "missing_income")]  = int(customers["annual_income_inr"].isnull().sum())

In [8]:
# Dates: acquisition_date has two formats, so try the main one, then fill the gaps with the other
main  = pd.to_datetime(customers["acquisition_date"], format="%d-%m-%Y", errors="coerce")
other = pd.to_datetime(customers["acquisition_date"], format="%Y/%m/%d", errors="coerce")
found[("customers", "mixed_date_format")] = int(main.isnull().sum())
customers["acquisition_date"] = main.fillna(other)

customers["date_of_birth"] = pd.to_datetime(customers["date_of_birth"], format="%d-%m-%Y", errors="coerce")
customers["churn_date"]    = pd.to_datetime(customers["churn_date"],    format="%d-%m-%Y", errors="coerce")

print("acquisition_date still unreadable:", customers["acquisition_date"].isnull().sum())
print(customers[["date_of_birth", "acquisition_date", "churn_date"]].dtypes)

acquisition_date still unreadable: 0
date_of_birth       datetime64[ns]
acquisition_date    datetime64[ns]
churn_date          datetime64[ns]
dtype: object


In [9]:
# Age: two checks. (1) outside 0-120. (2) not matching date_of_birth.
# (2) catches ages like 0 or 1 on adults, which (1) misses.
ref = pd.Timestamp("2026-03-31")                       # the data's "today"
true_age = (ref - customers["date_of_birth"]).dt.days / 365.25

out_of_range = (customers["age"] < 0) | (customers["age"] > 120)
not_matching = (customers["age"] - true_age).abs() > 1.5
bad_age = out_of_range | not_matching

print("Outside 0-120:", out_of_range.sum(), "| Not matching date_of_birth:", not_matching.sum())
print(customers.loc[bad_age, "age"].value_counts())

found[("customers", "impossible_age")] = int(bad_age.sum())
customers.loc[bad_age, "age"] = None

Outside 0-120: 23 | Not matching date_of_birth: 38
age
 1      12
 220     9
 130     8
-5       6
 0       3
Name: count, dtype: int64


In [10]:
# Phone and PIN. pandas read phones as numbers and dropped the "+", so a valid phone has 12 digits.
bad_phone = customers["phone_number"].astype(str).str.len() != 12
bad_pin   = customers["pin_code"].astype(str).str.len() != 6

found[("customers", "invalid_phone")] = int(bad_phone.sum())
found[("customers", "incorrect_pin")] = int(bad_pin.sum())

customers.loc[bad_phone, "phone_number"] = None
customers.loc[bad_pin,   "pin_code"]     = None

In [11]:
# City names: split the problem into ALL CAPS vs misspelled, then fix both with a city_id lookup
cities = tables["cities"]
true_city = customers["city_id"].map(cities.set_index("city_id")["city_name"])
print("city_ids not found in cities.csv:", true_city.isnull().sum())

wrong_city = customers["city_name"] != true_city
caps_only  = wrong_city & (customers["city_name"].str.lower() == true_city.str.lower())
misspelled = wrong_city & ~caps_only

found[("customers", "inconsistent_capitalisation")] = int(caps_only.sum())
found[("customers", "misspelled_city")]              = int(misspelled.sum())
print("Wrong in total:", wrong_city.sum(), "| capitals only:", caps_only.sum(), "| misspelled:", misspelled.sum())

customers["city_name"] = true_city

city_ids not found in cities.csv: 0
Wrong in total: 940 | capitals only: 560 | misspelled: 380


In [12]:
# Emails: a real email contains "@". Blank emails are a different problem (missing), so skip them here.
bad_email = customers["email"].notnull() & ~customers["email"].str.contains("@", na=False)
found[("customers", "invalid_email")] = int(bad_email.sum())
print(customers.loc[bad_email, "email"].head())
customers.loc[bad_email, "email"] = None

203    user544_at_mail
268    user123_at_mail
481    user615_at_mail
717    user428_at_mail
822    user690_at_mail
Name: email, dtype: object


In [13]:
# churn_date should be blank exactly for Active customers
print("Active:", (customers["customer_status"] == "Active").sum(),
      "| blank churn_date:", customers["churn_date"].isnull().sum())

final_nulls = customers.isnull().sum()
print(final_nulls[final_nulls > 0])
print("Final shape:", customers.shape)

customers.to_csv(DATA + "customers_clean.csv", index=False)
print("Saved customers_clean.csv")

Active: 15494 | blank churn_date: 15494
age                     38
email                 1029
phone_number           228
address                380
pin_code               190
annual_income_inr      950
churn_date           15494
dtype: int64
Final shape: (19000, 25)
Saved customers_clean.csv


### 3B. billing

In [14]:
billing = tables["billing"].copy()

found[("billing", "duplicate_invoices")] = int(billing.duplicated().sum())
billing = billing.drop_duplicates()

billing["billing_date"] = pd.to_datetime(billing["billing_date"], format="%d-%m-%Y", errors="coerce")

future   = billing["billing_date"] > "2026-03-31"          # the data ends in March 2026
negative = billing["total_amount_inr"] < 0
correct_gst = (billing["base_amount_inr"] * 0.18).round(2)
wrong_gst   = (billing["gst_amount_inr"] - correct_gst).abs() > 0.05

found[("billing", "future_dates")]            = int(future.sum())
found[("billing", "negative_invoice_amount")] = int(negative.sum())
found[("billing", "incorrect_gst")]           = int(wrong_gst.sum())

In [15]:
# Fixes: minus sign off (assumes sign errors, not refunds), GST recalculated, future invoices removed
billing.loc[negative, "total_amount_inr"] = billing.loc[negative, "total_amount_inr"].abs()
billing.loc[wrong_gst, "gst_amount_inr"]  = correct_gst[wrong_gst]
billing = billing[~future]

# Sanity check: after the fixes, does total = base + GST?
ok = ((billing["total_amount_inr"] - (billing["base_amount_inr"] + billing["gst_amount_inr"])).abs() < 0.05)
print("Rows where total = base + GST:", round(ok.mean() * 100, 2), "%")

print("Clean billing rows:", billing.shape)
billing.to_csv(DATA + "billing_clean.csv", index=False)

Rows where total = base + GST: 100.0 %
Clean billing rows: (114740, 9)


### 3C. payments

In [16]:
payments = tables["payments"].copy()

found[("payments", "duplicate_payments")] = int(payments.duplicated().sum())
payments = payments.drop_duplicates()

# Broken link = invoice_id that never existed in the ORIGINAL billing file
broken = ~payments["invoice_id"].isin(tables["billing"]["invoice_id"])
found[("payments", "broken_foreign_key")] = int(broken.sum())

# Separate group: invoices that existed but we removed (future-dated). Report this as an extra.
linked_to_removed = ~payments["invoice_id"].isin(billing["invoice_id"]) & ~broken
print("Payments on invoices we removed (future-dated):", linked_to_removed.sum())

payments = payments[payments["invoice_id"].isin(billing["invoice_id"])]
print("Clean payments rows:", payments.shape)
payments.to_csv(DATA + "payments_clean.csv", index=False)

Payments on invoices we removed (future-dated): 198
Clean payments rows: (91704, 7)


### 3D. subscriptions

In [17]:
subs = tables["subscriptions"].copy()

found[("subscriptions", "null_foreign_key")] = int(subs["plan_id"].isnull().sum())

unknown = subs["subscription_status"] == "UNKNOWN_STATE"
found[("subscriptions", "invalid_category")] = int(unknown.sum())

# PROVE the rule before using it: on the normal rows, does status always follow end_date?
known = subs[~unknown]
print(pd.crosstab(known["subscription_status"], known["end_date"].notnull()))

end_date             False  True 
subscription_status              
Active               16930      0
Terminated               0   3491


In [18]:
# The table above shows: Active = never has an end_date, Terminated = always has one. So:
has_end = subs["end_date"].notnull()
subs.loc[unknown & has_end,  "subscription_status"] = "Terminated"
subs.loc[unknown & ~has_end, "subscription_status"] = "Active"

print(subs["subscription_status"].value_counts())
subs.to_csv(DATA + "subscriptions_clean.csv", index=False)

# Heads-up for Phase 2/3: some customers hold more than one subscription
per_customer = subs.groupby("customer_id").size()
print("Customers with 2+ subscriptions:", (per_customer > 1).sum(), "| subscription rows:", len(subs))

subscription_status
Active        17017
Terminated     3506
Name: count, dtype: int64
Customers with 2+ subscriptions: 1523 | subscription rows: 20523


### 3E. usage_data (outliers)

In [19]:
usage = tables["usage_data"].copy()
gb = usage["data_gb_used"]

print(gb.describe().round(1))
print("\nLargest 'normal-looking' values, then where it jumps:")
print(gb[gb > 250].sort_values().head(20).round(0).tolist())

count    57000.0
mean        40.2
std        332.6
min          0.2
25%         11.4
50%         19.1
75%         32.6
max      22172.7
Name: data_gb_used, dtype: float64

Largest 'normal-looking' values, then where it jumps:
[251.0, 253.0, 256.0, 257.0, 258.0, 260.0, 262.0, 267.0, 276.0, 276.0, 280.0, 287.0, 293.0, 298.0, 308.0, 346.0, 357.0, 370.0, 372.0, 438.0]


In [20]:
# Cutoff: most customers use under ~150 GB and the tail thins out near 300.
# There is NO clean gap between 300 and 350, so the cutoff is a judgment call:
#   > 300 GB flags 173 rows, > 350 GB flags 171 (the number in the issue log).
# We use 350 and say so in the report.
OUTLIER_GB = 350

outlier = gb > OUTLIER_GB
found[("usage_data", "outlier_values")] = int(outlier.sum())
print("Outliers flagged (>", OUTLIER_GB, "GB):", outlier.sum(), "| would be", (gb > 300).sum(), "at 300 GB")

usage.loc[outlier, "data_gb_used"] = None
usage.to_csv(DATA + "usage_data_clean.csv", index=False)

Outliers flagged (> 350 GB): 171 | would be 173 at 300 GB


### 3F. complaints (share unresolved)

In [21]:
complaints = tables["complaints"]
status_share = (complaints["status"].value_counts(normalize=True) * 100).round(2)
print(status_share)
unresolved_pct = status_share["Unresolved"]

status
Resolved      55.69
Unresolved    44.31
Name: proportion, dtype: float64


## 4. Referential integrity: do all customer_ids exist in customers?

In [22]:
valid_ids = set(tables["customers"]["customer_id"])
ri = {}
for n, t in tables.items():
    if n != "customers" and "customer_id" in t.columns:
        ri[n] = int((~t["customer_id"].isin(valid_ids)).sum())
integrity = pd.Series(ri, name="unknown_customer_ids")
integrity

,unknown_customer_ids
billing,0
complaints,0
contracts,0
customer_feedback,0
devices,0
network_quality,0
payments,0
plan_history,0
recharges,0
retention_campaigns,0


## 5. Data Quality Report: my results vs the issue log (built by code)

In [23]:
log = tables["data_quality_issue_log"].copy()
log["found"] = [found.get((t, i)) for t, i in zip(log["table"], log["issue_type"])]
log["difference"] = log["found"] - log["rows_affected"]
log["match"] = log["difference"] == 0

n_match = int(log["match"].sum())
print(f"{n_match} of {len(log)} issues match the log exactly\n")
log

18 of 20 issues match the log exactly



,table,issue_type,rows_affected,found,difference,match
0,customers,missing_email,760,744,-16,False
1,customers,missing_address,380,380,0,True
2,customers,missing_income,950,950,0,True
3,customers,invalid_phone,228,228,0,True
4,customers,invalid_email,285,285,0,True
5,customers,incorrect_pin,190,190,0,True
6,customers,misspelled_city,380,380,0,True
7,customers,inconsistent_capitalisation,570,560,-10,False
8,customers,impossible_age,38,38,0,True
9,customers,mixed_date_format,380,380,0,True


In [24]:
# Key points, with every number pulled from the analysis above
active = int((customers["customer_status"] == "Active").sum())
print(f"- {len(tables)} tables loaded and profiled.")
print(f"- {n_match} of {len(log)} logged issues matched exactly; {len(log) - n_match} differ (see 'difference').")
print(f"- Customers: {len(tables['customers'])} raw rows -> {len(customers)} after removing duplicates.")
print(f"- churn_date is blank for all {customers['churn_date'].isnull().sum()} Active customers ({active} Active). Expected, not an error.")
print(f"- {unresolved_pct}% of complaints are Unresolved (a lead to test in Phase 2, not a proven churn driver).")
print(f"- Payments removed because their invoice was future-dated: {int(linked_to_removed.sum())}.")
print(f"- Unknown customer_ids across all tables: {int(integrity.sum())}.")

- 24 tables loaded and profiled.
- 18 of 20 logged issues matched exactly; 2 differ (see 'difference').
- Customers: 19076 raw rows -> 19000 after removing duplicates.
- churn_date is blank for all 15494 Active customers (15494 Active). Expected, not an error.
- 44.31% of complaints are Unresolved (a lead to test in Phase 2, not a proven churn driver).
- Payments removed because their invoice was future-dated: 198.
- Unknown customer_ids across all tables: 0.


## 6. Self-test: do the saved clean files really contain the fixes?

In [25]:
c = pd.read_csv(DATA + "customers_clean.csv")
b = pd.read_csv(DATA + "billing_clean.csv", parse_dates=["billing_date"])
p = pd.read_csv(DATA + "payments_clean.csv")
s = pd.read_csv(DATA + "subscriptions_clean.csv")
u = pd.read_csv(DATA + "usage_data_clean.csv")
cities_ref = pd.read_csv(DATA + "cities.csv")

tests = {
    "customers: 19,000 rows, no duplicate IDs":     len(c) == 19000 and c["customer_id"].is_unique,
    "customers: 38 blank ages":                      c["age"].isnull().sum() == 38,
    "customers: all city names valid":               c["city_name"].isin(cities_ref["city_name"]).all(),
    "customers: no email without @":                 not (c["email"].notnull() & ~c["email"].str.contains("@", na=False)).any(),
    "billing: no future dates":                      (b["billing_date"] <= "2026-03-31").all(),
    "billing: no negative totals":                   (b["total_amount_inr"] >= 0).all(),
    "billing: invoice IDs unique":                   b["invoice_id"].is_unique,
    "payments: every invoice exists in billing":     p["invoice_id"].isin(b["invoice_id"]).all(),
    "subscriptions: no UNKNOWN_STATE":               (s["subscription_status"] != "UNKNOWN_STATE").all(),
    "usage: no value above 350 GB":                  (u["data_gb_used"].dropna() <= 350).all(),
    "usage: 171 blank values":                       u["data_gb_used"].isnull().sum() == 171,
}
for name, ok in tests.items():
    print("PASS" if ok else "FAIL", "-", name)

PASS - customers: 19,000 rows, no duplicate IDs
PASS - customers: 38 blank ages
PASS - customers: all city names valid
PASS - customers: no email without @
PASS - billing: no future dates
PASS - billing: no negative totals
PASS - billing: invoice IDs unique
PASS - payments: every invoice exists in billing
PASS - subscriptions: no UNKNOWN_STATE
PASS - usage: no value above 350 GB
PASS - usage: 171 blank values


## 7. Data Quality Report: summary of findings

### What I checked
I loaded all 24 NexaTel tables and profiled each one: rows, columns, data types, blank values and duplicate rows. I then checked date formats, unusual values, and whether every customer_id in the other tables exists in the customers table, and compared my results with the issue log (table above).

### What I found
- **18 of the 20 logged issues matched the log exactly.**
- **2 did not match.** Missing emails: I found 744, the log says 760. Capitalisation errors: I found 560, the log says 570. I checked for hidden blanks (empty text, "N/A", "null") and found none, so I cannot explain these gaps. I am reporting them as not reproducible.
- **Duplicate rows exist in only 3 of the 24 tables:** customers (76), billing (344) and payments (276). All three match the log.
- **Ages:** 38 customers had impossible ages. 23 were outside 0-120 (values like 220, 130 and -5). 15 more were adults recorded as age 0 or 1. My first check (0-120 only) missed those 15, so I added a check against date_of_birth, which found all 38.
- Every customer_id in the other tables exists in customers.
- 44.31% of complaints are Unresolved.
- churn_date is blank for all 15,494 Active customers. This is expected, not an error.

### What I did about each issue
- **Removed duplicates:** 76 customers, 344 invoices, 276 payments.
- **Blanked, not deleted:** impossible ages (38), invalid phones (228), PINs (190), emails (285) and usage outliers (171). The customers' other data is valid, so I kept them.
- **Fixed:** 940 city names (560 ALL CAPS, 380 misspelled) using city_id; 380 mixed date formats; 1,149 wrong GST values (recalculated at 18%); 229 negative invoice totals.
- **Removed:** 229 invoices dated after March 2026 and the 198 payments linked to them, plus 184 payments pointing to invoices that do not exist.
- **Subscriptions:** 102 invalid statuses were set using end_date. I first checked that on all other rows, Active never has an end_date and Terminated always does. 41 missing plan_ids were left blank because they cannot be guessed.
- **Left blank on purpose:** email (744), address (380) and income (950). These cannot be guessed, and inventing values would distort the analysis.

### Assumptions
1. Negative invoice totals are sign errors, not refunds.
2. Invoices dated after 31 March 2026 are errors, because the data ends there.
3. Data usage above 350 GB per month is an outlier. There was no clean break between 300 and 350 GB (a 300 cutoff flags 173 rows, 350 flags 171). I chose 350 because it also matches the issue log, so this is a judgment call.

### What matters for the next phases
- Unresolved complaints are common (44.31%). This is something to test against churn, not a proven cause.
- 1,523 customers hold 2 subscriptions, so counts must use unique customers, not rows.
- Blanks left in age, email and income may affect some charts.

## File check: the 5 cleaned files saved in Drive

The cell below lists the cleaned files saved in the `NexaTel/data` folder with their save times (in UTC), and counts the files in the folder (24 original CSVs, 5 cleaned CSVs and the 4 project documents).

In [26]:
!ls -l --time-style=long-iso "$DATA" | grep clean
!ls "$DATA" | wc -l

-rw------- 1 root root 8994764 2026-10-29 09:22 billing_clean.csv
-rw------- 1 root root 4753582 2026-10-29 09:22 customers_clean.csv
-rw------- 1 root root 6110398 2026-10-29 09:22 payments_clean.csv
-rw------- 1 root root 1216549 2026-10-29 09:22 subscriptions_clean.csv
-rw------- 1 root root 2155247 2026-10-29 09:22 usage_data_clean.csv
33
